In [1]:
from pyspark.sql import SparkSession 
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, sum, mean , min , max , count , lit, lower, upper, initcap
from datetime import date , datetime

from functools import reduce                             

In [2]:
spark = SparkSession.builder.getOrCreate()

In [3]:
martBranches = spark.read.csv("mart_branches.csv" , header = 'True')
martCust = spark.read.csv("mart_customers.csv", header = 'True')
martProd = spark.read.csv("mart_products.csv", header = 'True')
martProm = spark.read.csv("mart_promotions.csv", header = 'True')
martSales = spark.read.csv("mart_sales_master.csv", header = 'True')

In [4]:
'''
now doing : martBranches
'''
martBranches.show() # print dataframe

+-----------+--------------+
|branch_code|   branch_name|
+-----------+--------------+
|         PJ| Petaling Jaya|
|         SB|   Subang Jaya|
|         KL|  Kuala Lumpur|
|         CH|        Cheras|
|         IP|          Ipoh|
|         PG|        Penang|
|         JB|   Johor Bahru|
|         PJ|Petaling Jayaa|
|         KL|  Kuala Lumper|
+-----------+--------------+



In [5]:
martBranches.printSchema()  
##dual string is fine

root
 |-- branch_code: string (nullable = true)
 |-- branch_name: string (nullable = true)



In [6]:
martBranches = martBranches.dropDuplicates(["branch_code"])
martBranches.show()
#no more duplicates

+-----------+-------------+
|branch_code|  branch_name|
+-----------+-------------+
|         CH|       Cheras|
|         IP|         Ipoh|
|         JB|  Johor Bahru|
|         KL| Kuala Lumpur|
|         PG|       Penang|
|         PJ|Petaling Jaya|
|         SB|  Subang Jaya|
+-----------+-------------+



In [7]:
##MART CUST NOW
martCust.show(20)
##null values
total_rows_cust= martCust.count()
#for later usage

+-----------+------+----+-----------+------------+
|customer_id|gender| age|member_tier|member_since|
+-----------+------+----+-----------+------------+
|      C1000|     F|NULL|       Gold|  2021-12-15|
|      C1001|     M|  57|     Silver|  2021-09-28|
|      C1002|  NULL|  41|       Gold|  2022-06-25|
|      C1003|     F|  30|       Gold|  2022-12-26|
|      C1004|  NULL|  43|       Gold|  2024-06-18|
|      C1005|     M|  52|     Silver|  2023-10-24|
|      C1006|     F|  68|     Silver|  2022-10-26|
|      C1007|     F|  33|     Silver|  2021-12-21|
|      C1008|     F|  45|     Silver|  2023-11-05|
|      C1009|     F|  36|     Silver|  2021-04-09|
|      C1010|     F|  59|     Silver|  2021-06-07|
|      C1011|     M|  19|       Gold|  2022-12-09|
|      C1012|     M|  46|   Platinum|  2024-03-15|
|      C1013|     F|  30|       NULL|  2023-12-11|
|      C1014|     M|  68|     Silver|  2023-09-12|
|      C1015|     F|  62|       NULL|  2022-04-04|
|      C1016|  NULL|  42|     S

In [8]:
martCust.printSchema()
martCust.describe().show()
#change age to num
#change member_since to date

root
 |-- customer_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: string (nullable = true)
 |-- member_tier: string (nullable = true)
 |-- member_since: string (nullable = true)

+-------+-----------+------+------------------+-----------+------------+
|summary|customer_id|gender|               age|member_tier|member_since|
+-------+-----------+------+------------------+-----------+------------+
|  count|       1800|  1683|              1739|       1509|        1800|
|   mean|       NULL|  NULL| 45.64059804485336|       NULL|        NULL|
| stddev|       NULL|  NULL|20.751258142261786|       NULL|        NULL|
|    min|      C1000|     F|               150|       Gold|  2021-01-01|
|    max|      C2799|     M|                69|     Silver|  2025-01-01|
+-------+-----------+------+------------------+-----------+------------+



In [9]:
martCust = martCust.withColumn("age", col("age").cast("integer"))
martCust = martCust.withColumn("member_since", col("member_since").cast("date"))
martCust.printSchema()
#types change , something something

root
 |-- customer_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- member_tier: string (nullable = true)
 |-- member_since: date (nullable = true)



In [10]:
distinct = martCust.distinct().count()
print(f"Duplicate rows: {total_rows_cust - distinct}")

Duplicate rows: 0


In [11]:

#checking skew + quartiles
martCust.select(
    F.skewness("age").alias("age_skew")
).show()
martCust.select(
    F.percentile_approx("age", [0, 0.25, 0.5, 0.75, 1]).alias("quartiles")
).show(truncate=False)

#150 :) 

#used to be below checking null , now is above , if shit dont make sense @me

+------------------+
|          age_skew|
+------------------+
|2.1757073389747315|
+------------------+

+---------------------+
|quartiles            |
+---------------------+
|[18, 30, 45, 57, 150]|
+---------------------+



In [12]:

print(martCust.filter("age = 150").count())
#why are there 32 ppl 150 age 
martCust = martCust.withColumn(
    "age",
    F.when(
        (F.col("age") >= 150),
        None
    ).otherwise(F.col("age"))
)

print(martCust.filter("age = 150").count())

32
0


In [13]:
martCust.select(
    F.skewness("age").alias("age_skew")
).show()
martCust.select(
    F.percentile_approx("age", [0, 0.25, 0.5, 0.75, 1]).alias("quartiles")
).show(truncate=False)
#surely its fixed now, seems good enough

+--------------------+
|            age_skew|
+--------------------+
|-0.04500051970636024|
+--------------------+

+--------------------+
|quartiles           |
+--------------------+
|[18, 30, 45, 57, 69]|
+--------------------+



In [14]:
martCust.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in martCust.columns
]).show()

#count no. null values

+-----------+------+---+-----------+------------+
|customer_id|gender|age|member_tier|member_since|
+-----------+------+---+-----------+------------+
|          0|   117| 93|        291|           0|
+-----------+------+---+-----------+------------+



In [15]:
martCust.select([
    (
        F.count(F.when(F.col(c).isNull(), c))
        / total_rows_cust * 100
    ).alias(c)
    for c in martCust.columns
]).show()

#null value percentage
#so the question is : what does null values in member_tier mean
#either no data , or  not a tier , i dont think any other attribute correlates to member tier 
#however sales_master does refrence customer id 
#so plan is to not drop any rows
#fill gender with Mode , Age with Median , replace null in membertier with Unmembered
#other option is to drop member_tier directly , does only show up in martCust

+-----------+------+-----------------+------------------+------------+
|customer_id|gender|              age|       member_tier|member_since|
+-----------+------+-----------------+------------------+------------+
|        0.0|   6.5|5.166666666666667|16.166666666666664|         0.0|
+-----------+------+-----------------+------------------+------------+



In [16]:
#fill gender with Mode
mode_val = martCust.groupBy("gender").count().orderBy("gender", ascending=False).first()[0]
martCust = martCust.fillna({"gender": mode_val})

In [17]:
#fill age with mean ,ttfs
mean_val = martCust.select(mean("age")).first()[0]
martCust = martCust.fillna({"age": mean_val})

#fill member_tier nulls with Unmembered
martCust = martCust.fillna({"member_tier": "Unmembered"})

In [18]:
#I will assume Null means Basic Member
#But in the case of needing to drop

#martCust = masrtCust.dropna(subset=["member_tier"])


In [19]:
martCust.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in martCust.columns
]).show()
# only member tier remains
#fuck i did not forgot duplicates

+-----------+------+---+-----------+------------+
|customer_id|gender|age|member_tier|member_since|
+-----------+------+---+-----------+------------+
|          0|     0|  0|        291|           0|
+-----------+------+---+-----------+------------+



In [20]:
#final checks to be safe
martCust.groupBy("gender").count().show()

martCust.filter(
    F.col("member_since") > F.lit("2026-06-01")
).show()

martCust.groupBy("customer_id") \
        .count() \
        .filter("count > 1") \
        .show()

#no inconsitencies , go next 

+------+-----+
|gender|count|
+------+-----+
|     F|  843|
|     M|  957|
+------+-----+

+-----------+------+---+-----------+------------+
|customer_id|gender|age|member_tier|member_since|
+-----------+------+---+-----------+------------+
+-----------+------+---+-----------+------------+

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [ ]:
# member_since bounds check: flag and null out invalid dates
# Program started 2021-01-01; future dates beyond today are impossible
print("member_since before 2021-01-01:", martCust.filter(F.col("member_since") < F.lit("2021-01-01")).count())
print("member_since after 2026-06-01:", martCust.filter(F.col("member_since") > F.lit("2026-06-01")).count())

# Null out out-of-range dates
martCust = martCust.withColumn(
    "member_since",
    F.when(
        (F.col("member_since") < F.lit("2021-01-01")) | (F.col("member_since") > F.lit("2026-06-01")),
        None
    ).otherwise(F.col("member_since"))
)

# Verify
print("member_since nulls after clean:", martCust.filter(F.col("member_since").isNull()).count())


In [21]:
## MART PROD
martProd.show()

total_rows_prod = martProd.count()
print (total_rows_prod)
## so , basic data checking
#no null , unqiue product_ids , categoris are fine , price is fine 
#also dont see any dupes
#just datatypes imo

+----------+--------------------+--------+---------+----------+
|product_id|        product_name|category|std_price|cost_price|
+----------+--------------------+--------+---------+----------+
|    BEV001|      Coca Cola 1.5L|Beverage|      4.5|       3.8|
|    BEV002| Mineral Water 500ml|Beverage|      1.8|       0.6|
|    BEV003|     Orange Juice 1L|Beverage|      7.9|       5.9|
|    SNK001|    Potato Chips BBQ|   Snack|      6.9|       3.0|
|    SNK002|Chocolate Cookies...|   Snack|      8.2|       4.6|
|    HOM001|       Detergent 2kg|    Home|     18.9|      15.2|
|    HOM002|Dishwashing Liqui...|    Home|      9.5|       6.1|
|    FRS001|      Apple Fuji 1kg|   Fresh|      8.5|       7.0|
|    FRS002|          Banana 1kg|   Fresh|      5.5|       4.8|
|    PER001|       Shampoo 650ml|Personal|     15.9|      10.4|
|    PER002|        Body Wash 1L|Personal|     14.2|       9.7|
|    FRZ001|  Chicken Nugget 1kg|  Frozen|     14.9|      11.8|
|    SNK001|Potato Chips Barb...|   Snac

In [22]:
distinct = martProd.distinct().count()
print(f"Duplicate rows: {total_rows_prod - distinct}")
#be safe?

Duplicate rows: 0


In [23]:
martProd.printSchema()  

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- std_price: string (nullable = true)
 |-- cost_price: string (nullable = true)



In [24]:
martProd = martProd.withColumn("std_price", col("std_price").cast("float"))
martProd = martProd.withColumn("cost_price", col("cost_price").cast("float"))

In [25]:
martProd.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- std_price: float (nullable = true)
 |-- cost_price: float (nullable = true)



In [26]:
## MART PROMOTION
martProm.show()

#realisitcally , do nothing
#only whitespace inconsitencies?

# null check
martProm.select([count(when(col(c).isNull(), c)).alias(c) for c in martProm.columns]).show()
# dupe check
print("Prom dupes:", martProm.count() - martProm.distinct().count())
# drop BADPROMO from reference table
martProm = martProm.filter(F.col("promo_code") != "BADPROMO")

+----------+--------------------+
|promo_code|         description|
+----------+--------------------+
|      NONE|            No Promo|
|   MEMBER8|           Member 8%|
|     CNY10|             CNY 10%|
|YEAR_END15|        Year End 15%|
|    LOSS30|     Loss Leader 30%|
|  BADPROMO|Corrupted Promo Code|
+----------+--------------------+



In [27]:
## MART SALES

martSales.show()

total_rows_sales = martSales.count()
print(total_rows_sales)
#das alot of rows 71654

##to do ensure datypes are valid

+----------+-------------------+------------+----------+--------------------+---+----------+----------+-----------+
|invoice_id|            txn_raw|      branch|product_id|        product_name|qty|unit_price|promo_code|customer_id|
+----------+-------------------+------------+----------+--------------------+---+----------+----------+-----------+
| INV100000|   19/11/2025 21:34| PetalingJya|    SNK002|ChocolateCookies200g|  3|      7.38|     CNY10|      C1220|
| INV100001|   08-03-2025 10:32| johor bahru|    BEV001|           Coke 1.5L|  2|       4.5|      NONE|      C1680|
| INV100002|2025-08-13 18:19:00|KUALA LUMPUR|    BEV001|        CocaCola1.5L|  1|      4.05|     CNY10|      C2148|
| INV100003|2025-11-02 16:03:00|KUALA LUMPUR|    FRZ001|  Chicken Nugget 1kg|  2|     13.71|   MEMBER8|      C1893|
| INV100004|   06-26-2025 16:41|          JB|    BEV002| Mineral Water 500ml|  2|       1.8|      NONE|      C1646|
| INV100005|   02/08/2025 15:30|          PG|    FRS001|      Apple Fuji

In [28]:
martSales.printSchema()
# so 
# invoice - fine
# txn - date time fix formatting
# branch - string fix formatting
# prod_id -fine
# name - fine , check ending seq
# qty - CHANGE int
# price CHANGE float
# code fine
# custID - fine

root
 |-- invoice_id: string (nullable = true)
 |-- txn_raw: string (nullable = true)
 |-- branch: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- qty: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- promo_code: string (nullable = true)
 |-- customer_id: string (nullable = true)



In [29]:
#restucturing Stuff Doing : Invoice ID first
#string is fine 
#check dupes


martSales.groupBy("invoice_id") \
        .count() \
        .filter("count > 1") \
        .show()

martSales.filter("invoice_id = 'INV112753'").show()
martSales.filter("invoice_id = 'INV110049'").show()



#wtf do maths?
#OJ 1l = 7.9
#so math is total * discount / sum 
#so 7.9 * 5 / *0.98 / 5 = 7.742 ???
#so 7.9 * 10 / *0.98 / 10  = 7.742 ???
# cs - std ? 
# (10(7.9) - 10(5.9))*98%
#???

+----------+-----+
|invoice_id|count|
+----------+-----+
| INV101656|    2|
| INV117957|    2|
| INV124372|    2|
| INV125692|    2|
| INV135717|    2|
| INV112753|    2|
| INV134599|    2|
| INV148466|    2|
| INV109683|    2|
| INV136774|    2|
| INV128725|    2|
| INV139497|    2|
| INV146915|    2|
| INV109804|    2|
| INV121485|    2|
| INV125351|    2|
| INV131682|    2|
| INV137084|    2|
| INV110797|    2|
| INV132461|    2|
+----------+-----+
only showing top 20 rows

+----------+-----------------+------+----------+---------------+---+----------+----------+-----------+
|invoice_id|          txn_raw|branch|product_id|   product_name|qty|unit_price|promo_code|customer_id|
+----------+-----------------+------+----------+---------------+---+----------+----------+-----------+
| INV112753|Nov 08 2025 16:33|Cheras|    BEV003|ORANGE JUICE 1L|  5|      7.27|   MEMBER8|      C2147|
| INV112753|Nov 08 2025 16:33|Cheras|    BEV003|ORANGE JUICE 1L|  7|      10.9|   MEMBER8|      C2147|
+--

In [30]:
#invoice ID NUlls
#go next

In [31]:
#TXN raw

##checking for null first
#data type shenneingans will check other later
martSales.select(
    F.count(
        F.when(F.col("txn_raw").isNull(), 1)
    ).alias("txn_raw_nulls")
).show()

+-------------+
|txn_raw_nulls|
+-------------+
|          559|
+-------------+



In [32]:
# txn_raw

#19/11/2025 21:34
#dd/MM/yyyy HH:mm

#2025-11-02 16:03:00
#yyyy-MM-dd HH:mm:ss

#Apr 14 2025 18:34
#MMM dd yyyy HH:mm

#06-26-2025 16:41
#MM-dd-yyyy HH:mm

#2025-11-19 21:34:00
#yyyy-MM-dd HH:mm:ss

#31/02/2025 10:00
#ddMMyyyy HH:mm



##martSales = martSales.withColumn("txn_raw", col("txn_raw").cast("timestamp"))

#just checking if txn_has values that nulled

##validating dates all are 

## so turn all to null

print( "Nulls:"  ,
    martSales.filter(
        F.col("txn_raw").isNull()
    ).count()
)

"""martSales = martSales.withColumn(
    "txn_raw",
    F.when(F.col("txn_raw") == "NULL", None).otherwise(F.col("txn_raw"))
)"""

martSales = martSales.withColumn(
    "txn_cooked",
    F.coalesce(
        F.to_timestamp("txn_raw", "dd/MM/yyyy HH:mm"),
        F.to_timestamp("txn_raw", "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp("txn_raw", "MMM dd yyyy HH:mm"),
        F.to_timestamp("txn_raw", "MM-dd-yyyy HH:mm")
    
    )
)

martSales.filter(
    F.col("txn_cooked").isNull()
).groupBy("txn_raw").count().show(100, False)

print( "Nulls:"  ,
    martSales.filter(
        F.col("txn_cooked").isNull()
    ).count()
)
martSales = (
    martSales
    .drop("txn_raw")
    .withColumn("transaction_datetime" , F.col("txn_cooked"))
    .drop("txn_cooked")
)

martSales.show()

Nulls: 559
+----------------+-----+
|txn_raw         |count|
+----------------+-----+
|31/02/2025 10:00|61   |
|NULL            |559  |
|notadate        |56   |
|2025-99-01      |50   |
|32/13/2025      |53   |
|yesterday       |71   |
|???             |1    |
|2025/88/12      |1    |
+----------------+-----+

Nulls: 852
+----------+------------+----------+--------------------+---+----------+----------+-----------+--------------------+
|invoice_id|      branch|product_id|        product_name|qty|unit_price|promo_code|customer_id|transaction DateTime|
+----------+------------+----------+--------------------+---+----------+----------+-----------+--------------------+
| INV100000| PetalingJya|    SNK002|ChocolateCookies200g|  3|      7.38|     CNY10|      C1220| 2025-11-19 21:34:00|
| INV100001| johor bahru|    BEV001|           Coke 1.5L|  2|       4.5|      NONE|      C1680| 2025-08-03 10:32:00|
| INV100002|KUALA LUMPUR|    BEV001|        CocaCola1.5L|  1|      4.05|     CNY10|      C21

In [33]:
##txn fill nulls
#should really define this as a function but oh well 
martSales.select([
    (
        F.count(F.when(F.col("transaction_datetime").isNull(), 1))
        / total_rows_sales * 100
    ).alias("null_pct")
]).show()

print( "NUlls:" ,
    martSales.filter(
        F.col("transaction_datetime").isNull()
    ).count()
)


mode_val = martSales.filter(F.col("transaction_datetime").isNotNull()) \
    .groupBy("transaction_datetime") \
    .count() \
    .orderBy("count", ascending=False) \
    .first()[0]

martSales = martSales.withColumn(
    "transaction_datetime",
    F.when(F.col("transaction_datetime").isNull(), F.lit(mode_val))
    .otherwise(F.col("transaction_datetime"))
)


print( "Nulls:"  ,
    martSales.filter(
        F.col("transaction_datetime").isNull()
    ).count()
)



##filling with mode since low pct 

+------------------+
|          null_pct|
+------------------+
|1.1890473665112904|
+------------------+

NUlls: 852
Nulls: 0


In [34]:
##Branch Time
#is string so is fine

#check nulls
martSales.select(
    F.count(
        F.when(F.col("branch").isNull(), 1)
    ).alias("branch nulls")
).show()

martSales.groupBy("branch").count().show(100, False)

# Branch standardisation using martBranches as the reference table
# Normalise both sides: strip whitespace, lowercase, remove spaces/dots
# then join to get the canonical branch_code

import re
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def normalise_str(s):
    if s is None:
        return None
    s = s.strip().lower()
    s = re.sub(r"[\s.\-_]+", "", s)
    return s

normalise_udf = udf(normalise_str, StringType())

# Build normalised lookup from martBranches
branch_lookup = martBranches.withColumn("branch_key", normalise_udf(F.col("branch_name")))     .select("branch_key", F.col("branch_code").alias("std_code"))

# Normalise sales branch column and join
martSales = martSales.withColumn("branch_key", normalise_udf(F.col("branch")))

martSales = martSales.join(branch_lookup, on="branch_key", how="left")     .drop("branch", "branch_key")     .withColumnRenamed("std_code", "branch")

# Null remaining unmatched values (ERROR, UNKNOWN, gibberish)
unmatched = martSales.filter(F.col("branch").isNull()).count()
print(f"Unmatched branch values (set to null): {unmatched}")

martSales.groupBy("branch").count().show(100, False)


+------------+
|branch nulls|
+------------+
|         801|
+------------+

+--------------+-----+
|branch        |count|
+--------------+-----+
|IPOH          |1963 |
|SB            |1398 |
|johor bahru   |1984 |
|Petaling Jaya |1450 |
|Petaling Jayaa|1497 |
|Subang        |1353 |
|Kuala Lumpur  |1256 |
|NULL          |801  |
|penang        |1934 |
|PG            |1975 |
|SubangJaya    |1382 |
|KL            |1193 |
|kuala lumpur  |1172 |
|SUBANG JAYA   |1356 |
|JohorBahru    |1938 |
|KualaLumpur   |2436 |
|KUALA LUMPUR  |1242 |
|K.L.          |1225 |
|subang jaya   |1411 |
|ipoh          |1955 |
|CH            |1965 |
|JB            |1925 |
|Penang        |3888 |
|PetalingJya   |1541 |
|PENANG        |1935 |
|PETALING JAYA |1510 |
|PetalingJaya  |1560 |
|Subang Jaya   |1331 |
|JOHOR BAHRU   |2013 |
|P.J.          |1504 |
|Subng Jaya    |1411 |
|PJ            |1548 |
|IP            |1963 |
|Johor Bahru   |2003 |
|CHERAS        |2014 |
|Cheras        |3881 |
|Ipoh          |3980 |
|Kua

In [35]:
## checking branch null pct
martSales.select([
    (
        F.count(F.when(F.col("branch").isNull(), 1))
        / total_rows_sales * 100
    ).alias("null_pct")
]).show()

mode_val = martSales.groupBy("branch").count().orderBy("count", ascending=False).first()[0]
martSales = martSales.fillna({"branch": mode_val})

martSales.groupBy("branch").count().show(100, False)

print( "Nulls:"  ,
    martSales.filter(
        F.col("branch").isNull()
    ).count()
)


+------------------+
|          null_pct|
+------------------+
|1.1206631869818853|
+------------------+

+------+-----+
|branch|count|
+------+-----+
|SB    |9642 |
|PG    |9732 |
|KL    |9753 |
|CH    |9839 |
|JB    |9863 |
|PJ    |12964|
|IP    |9861 |
+------+-----+

Nulls: 0


In [36]:
## qty  ##nulls #outliers # impossible value
#nulls will end up filling so remove impossible / outliers first 
martSales.groupBy("qty") \
        .count() \
        .show(10000)
# qty - CHANGE int
martSales = martSales.withColumn("qty", col("qty").cast("integer"))

martSales = martSales.withColumn(
    "qty",
    F.when(F.col("qty") < 0, None)
    .otherwise(F.col("qty"))
)

martSales.groupBy("qty") \
        .count() \
        .show(10000)


+----+-----+
| qty|count|
+----+-----+
|  -4|  149|
|  51|    1|
|  -1| 2135|
| two|   49|
| 101|    1|
|  -6|    2|
|  69|    2|
|  29|    2|
|  42|    1|
| -39|    2|
|   3|10544|
| 113|    1|
|  34|    1|
|  28|    2|
|  22|    2|
|  35|    1|
| -22|    3|
|  71|    1|
|  99|    2|
| 107|    1|
| -29|    3|
|  -8|    1|
|NULL|  744|
|   5| 1655|
| 100|    1|
|-999|   40|
|  27|    2|
|  75|    1|
|  46|    3|
| -23|    3|
| -10|    5|
|   6|  845|
| 118|    2|
| -25|    1|
| -41|    3|
|  68|    1|
|  90|    1|
| 104|    1|
|  41|    1|
| 102|    1|
| -42|    6|
| -19|    2|
| 111|    1|
| -12|    1|
|  95|    1|
| -32|    2|
|  81|    1|
| -16|    1|
| 114|    1|
| -44|    4|
| -43|    1|
|  48|    2|
| -35|    1|
| 1.5|   52|
|  67|    1|
|  84|    2|
|9999|   44|
|  79|    2|
|  24|    1|
|  88|    2|
|   1|28667|
| -11|    4|
| -18|    5|
|  36|    1|
| abc|   49|
|  37|    2|
| -20|    1|
|  49|    1|
| -33|    4|
|  -3|  154|
| -28|    4|
| -24|    2|
|  65|    3|
|   4| 7742|

In [37]:
martSales.select(
    F.skewness("qty").alias("qty_skew")
).show()
martSales.select(
    F.percentile_approx("qty", [0, 0.25, 0.5, 0.75, 1]).alias("quartiles")
).show(truncate=False)

martSales.select([
    (
        F.count(F.when(F.col("qty").isNull(), 1))
        / total_rows_sales * 100
    ).alias("null_pct")
]).show()

martSales = martSales.withColumn("qty",
    F.when(F.col("qty") >= 9999, None)
     .otherwise(F.col("qty"))
)

upper = martSales.approxQuantile("qty", [0.99], 0.01)[0]
martSales = martSales.withColumn(
    "qty_capped",
    F.when(F.col("qty") > upper, upper).otherwise(F.col("qty"))
)

martSales.select(
    F.skewness("qty_capped").alias("qty_capped_skew")
).show()
martSales.select(
    F.percentile_approx("qty_capped", [0, 0.25, 0.5, 0.75, 1]).alias("quartiles")
).show(truncate=False)

martSales.select([
    (
        F.count(F.when(F.col("qty_capped").isNull(), 1))
        / total_rows_sales * 100
    ).alias("null_pct")
]).show()

martSales.orderBy(F.col("qty_capped").desc()).show(100)

+-----------------+
|         qty_skew|
+-----------------+
|39.25321498855976|
+-----------------+

+------------------+
|quartiles         |
+------------------+
|[1, 1, 2, 3, 9999]|
+------------------+

+-----------------+
|         null_pct|
+-----------------+
|5.170681329723393|
+-----------------+



In [ ]:
# Fill qty nulls with median after outlier removal
median_qty = martSales.approxQuantile("qty", [0.5], 0.01)[0]
martSales = martSales.fillna({"qty": int(median_qty)})
print("qty nulls after fill:", martSales.filter(F.col("qty").isNull()).count())

In [ ]:
# price
martSales.groupBy("unit_price") \
        .count() \
        .show(10000)
##RM presumabably just remove it
# so remove RM and 
# free = 0

In [ ]:
# price CHANGE float

martSales = martSales.withColumn("unit_price",
    when(col("unit_price").startswith("RM"), col("unit_price").substr(3, 100).cast("float"))
    .when(col("unit_price") == "free", lit(0).cast("float"))
    .otherwise(col("unit_price").cast("float"))
)

martSales.groupBy("unit_price") \
        .count() \
        .show(10000)


# Check for any unparseable unit_price values that became null
print("unit_price nulls:", martSales.filter(F.col("unit_price").isNull()).count())

In [ ]:
martSales.printSchema()

In [ ]:
# unit_price sanity check: flag rows where unit_price > std_price (no promo should raise price)
price_check = martSales.join(
    martProd.select("product_id", "std_price"),
    on="product_id",
    how="left"
).withColumn(
    "price_above_std",
    F.col("unit_price") > F.col("std_price")
)

above = price_check.filter(F.col("price_above_std") == True)
print("Rows where unit_price > std_price:", above.count())
above.groupBy("promo_code").count().orderBy("count", ascending=False).show()


In [ ]:
##check dupe rows 
distinct = martSales.distinct().count()
print(f"Duplicate rows: {total_rows_sales - distinct}")

martSales = martSales.distinct()

print(f"After dedup: {martSales.count()} rows remaining")


In [ ]:
martSales.groupBy("branch") \
        .count() \
        .show()

In [ ]:
martSales.groupBy("product_id") \
        .count() \
        .show()
'''	product_id
1	BEV001
2	BEV002
3	BEV003
4	SNK001
5	SNK002
6	HOM001
7	HOM002
8	FRS001
9	FRS002
10	PER001
11	PER002
12	FRZ001
13	SNK001'''

In [ ]:
martSales.groupBy("product_name") \
        .count() \
        .show(100)
'''
merge on product id
'''
#martProd("product_id")
#martProd("product_name")
lookup = martProd.select("product_id", col("product_name").alias("prod_name_clean"))

martSales = martSales.join(lookup, on="product_id", how="left") \
       .withColumn("product_name", col("prod_name_clean")) \
       .drop("prod_name_clean")

martSales.groupBy("product_name") \
        .count() \
        .show(100)

In [ ]:
martSales.groupBy("promo_code") \
        .count() \
        .show()

''' expected values
1	NONE
2	MEMBER8
3	CNY10
4	YEAR_END15
5	LOSS30
6	BADPROMO
'''

In [ ]:
martSales.groupBy("customer_id") \
        .count() \
        .show(10000)

## check unqiue

In [ ]:
# Referential integrity checks
# Sales rows with no matching customer
missing_cust = martSales.join(martCust.select("customer_id"), on="customer_id", how="left_anti")
print("Sales rows with unknown customer_id:", missing_cust.count())

# Sales rows with no matching product
missing_prod = martSales.join(martProd.select("product_id"), on="product_id", how="left_anti")
print("Sales rows with unknown product_id:", missing_prod.count())

# Sales rows with no matching branch
missing_branch = martSales.join(martBranches.select("branch_code"), martSales.branch == martBranches.branch_code, how="left_anti")
print("Sales rows with unknown branch:", missing_branch.count())

# Sales rows with no matching promo_code
missing_promo = martSales.join(martProm.select("promo_code"), on="promo_code", how="left_anti")
print("Sales rows with unknown promo_code:", missing_promo.count())


In [ ]:
##should probably resturcture this 
## if comments are weird @ me

In [ ]:
# === DATA QUALITY SUMMARY ===
print("=" * 60)
print(f"{'Table':<20} {'Rows':>8}  {'Total Nulls':>12}")
print("-" * 60)

dfs = {
    "martBranches": martBranches,
    "martCust": martCust,
    "martProd": martProd,
    "martProm": martProm,
    "martSales": martSales,
}

for name, df in dfs.items():
    row_count = df.count()
    null_counts = df.select(
        [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]
    ).collect()[0]
    total_nulls = sum(null_counts)
    print(f"{name:<20} {row_count:>8}  {total_nulls:>12}")
    # Show per-column breakdown if there are any nulls
    if total_nulls > 0:
        for col_name in df.columns:
            n = null_counts[col_name]
            if n > 0:
                print(f"    {col_name}: {n} nulls")

print("=" * 60)


In [ ]:
# Save all cleaned dataframes
martSales.write.csv("mart_sales_cleaned", header=True, mode="overwrite")
martCust.write.csv("mart_customers_cleaned", header=True, mode="overwrite")
martProd.write.csv("mart_products_cleaned", header=True, mode="overwrite")
martProm.write.csv("mart_promotions_cleaned", header=True, mode="overwrite")
martBranches.write.csv("mart_branches_cleaned", header=True, mode="overwrite")
print("All cleaned files saved.")